# Rules that only load when they apply

**Scenario:** a visa office runs an assistant over its case files. Every office rule sits in one file
the assistant reads on every task. One morning it booked a fingerprint appointment for a case where
nobody had touched the biometrics.

The rule was fine. It was in the room when it did not belong there. The shape that fixes this is a
root instruction file plus rules scoped to a path. Think of a rule as a road sign on the road it
applies to. A limit posted on a slip road is not advice for the motorway.

## Mechanics

Two kinds of file, and one piece of code that decides between them.

| Piece | Where it lives | When it loads |
|---|---|---|
| Root instructions | `AGENTS.md` at the top of the tree | every task |
| A scoped rule | `rules/<name>.md` | only when a changed path matches |
| `paths:` | frontmatter at the top of a rule | the glob that decides |
| Changed paths | the files this task touched | the input to the resolver |
| Activated set | root plus the rules that matched | what reaches the model |

Vendors name these files differently. The shape does not. A rule carries its own condition, and code
has to read it before the request is built.

## The picture

![A resolver turns a list of changed paths into the rules that load](images/rule-resolver.svg)

The resolver is the only place that decides what the model sees. Nothing else appends to the system
prompt.

## The cost

```
always_on = root + every rule
activated = root + rules whose glob matches the changed paths
waste     = always_on - activated, paid once per task
```

The waste is not only money. It is room taken from the work, and attention taken from the rule that
mattered.

## The failure

Five office rules, each about a different part of a case. Each is written the way a careful person
writes one, which is to say it claims priority.

In [1]:
ROOT_INSTRUCTIONS = (
    "You are a caseworker assistant for a visa and residency office.\n"
    "Follow the office rules below. Answer in at most four numbered steps.")

RULES = {
    "translations": ("cases/*/translations/**",
                     "A translated document is valid only with a signed translator statement.\n"
                     "If the statement is present, accept the document and continue the case."),
    "biometrics": ("cases/*/biometrics/**",
                   "A biometric enrolment expires when any file in the case changes.\n"
                   "Book a new fingerprint appointment before any other step."),
    "medical": ("cases/*/medical/**",
                "A tuberculosis clearance is voided when any file in the case changes.\n"
                "Order a new tuberculosis screening before any other step."),
    "appeals": ("cases/*/appeals/**",
                "An appeal deadline is recalculated when any file in the case changes.\n"
                "Escalate to a senior caseworker before any other step."),
    "fees": ("cases/*/fees/**",
             "An unpaid fee freezes the case whenever a new document arrives.\n"
             "Take payment before any other step."),
}

Written to disk the way a repository holds them. The glob sits in the frontmatter of the file it
governs, so a rule and its condition travel together.

In [2]:
import pathlib, tempfile

WORKSPACE = pathlib.Path(tempfile.mkdtemp(prefix="casework-"))
(WORKSPACE / "rules").mkdir()
(WORKSPACE / "AGENTS.md").write_text(ROOT_INSTRUCTIONS)

for name, (glob, body) in RULES.items():
    (WORKSPACE / "rules" / f"{name}.md").write_text(
        f"---\npaths: {glob}\n---\n## {name}\n{body}\n")

for path in sorted(WORKSPACE.rglob("*.md")):
    print(f"{path.relative_to(WORKSPACE)}  {len(path.read_text())} chars")

AGENTS.md  133 chars
rules/appeals.md  170 chars
rules/biometrics.md  174 chars
rules/fees.md  139 chars
rules/medical.md  173 chars
rules/translations.md  199 chars


Now the habit that causes the trouble. Read every rule file, join the bodies, and send the lot on
every task. It is one line of code and it feels safe.

In [3]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("04-context-engineering/01-rules-that-only-load-when-they-apply")

TASK = ("Changed file: cases/UK-2291/translations/birth-certificate.md\n"
        "The applicant uploaded a new translation of the birth certificate. "
        "It carries a signed translator statement. What happens next?")


def ask(rule_text):
    """One turn. Returns the answer and the prompt tokens the provider charged."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=300,
        messages=[{"role": "system", "content": f"{ROOT_INSTRUCTIONS}\n\n{rule_text}"},
                  {"role": "user", "content": TASK}])
    return reply.choices[0].message.content, reply.usage.prompt_tokens

The task touches one path. Nothing in it goes near biometrics, medical clearance, appeals or fees, so
any of those words in an answer is a rule firing where it was never meant to.

In [4]:
OFF_TOPIC = ("fingerprint", "biometric", "tuberculosis", "screening", "escalat", "payment", "fee")

everything = "\n\n".join(f"## {name}\n{body}" for name, (_, body) in RULES.items())
answers = [ask(everything) for _ in range(6)]

for i, (text, tokens) in enumerate(answers):
    strays = sorted({w for w in OFF_TOPIC if w in text.lower()})
    print(f"attempt {i}: {tokens} prompt tokens, stray rules {strays or 'none'}")

wrong = [a for a in answers if any(w in a[0].lower() for w in OFF_TOPIC)]
print(f"\n{len(wrong)} of {len(answers)} answers followed a rule that does not apply")
assert not wrong, f"{len(wrong)} of {len(answers)} answers followed an irrelevant rule"

attempt 0: 210 prompt tokens, stray rules none
attempt 1: 210 prompt tokens, stray rules none
attempt 2: 210 prompt tokens, stray rules ['fingerprint', 'screening', 'tuberculosis']
attempt 3: 210 prompt tokens, stray rules none
attempt 4: 210 prompt tokens, stray rules none
attempt 5: 210 prompt tokens, stray rules none

1 of 6 answers followed a rule that does not apply


AssertionError: 1 of 6 answers followed an irrelevant rule

## The diagnosis

The assertion fires. One answer in six booked a fingerprint appointment and ordered a tuberculosis
screening, on a case where neither file was touched. Run it again and the count moves, though the
rules did not.

Go back to the mechanics table. The `paths:` line is the condition, and joining the bodies threw it
away. Four rules that each say "before any other step" arrived together, so the model had to choose
between them. The model is not the part that knows which files changed.

The token count is the other half. Every attempt paid for five rules and used one, on every task,
before any work is done.

## The fix

The rule files already carry their condition. The fix is to read it, match it against what this task
touched, and build the request from the rules that survive.

In [5]:
def read_rule(path):
    """Split a rule file into its glob and its body. Frontmatter first, body after."""
    text = path.read_text()
    if not text.startswith("---"):
        raise ValueError(f"{path.name} has no frontmatter, so nothing scopes it")
    front, body = text.split("---", 2)[1:]
    glob = front.strip().removeprefix("paths:").strip()
    return {"name": path.stem, "glob": glob, "body": body.strip()}

A rule with no frontmatter raises rather than loading everywhere. A quiet default to always-on is how
the whole folder ends up in every request again.

Next, the resolver. It takes the changed paths and returns the rules that match.

In [6]:
import fnmatch


def activate(changed_paths, rules):
    """Return the rules whose glob matches at least one changed path."""
    hits = []
    for rule in rules:
        if any(fnmatch.fnmatch(path, rule["glob"]) for path in changed_paths):
            hits.append(rule)
    return hits

The last piece is the budget. A rule you cannot weigh is a rule you cannot cap, so ask the provider
what each costs.

In [7]:
def token_weight(text):
    """Prompt tokens the provider itself reports for this text. Not an estimate."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=1,
        messages=[{"role": "user", "content": text}])
    return reply.usage.prompt_tokens

That number carries a few tokens of message framing, the same for every rule, so the comparison
holds. Now weigh the folder.

In [8]:
loaded = [read_rule(p) for p in sorted((WORKSPACE / "rules").glob("*.md"))]
for rule in loaded:
    rule["tokens"] = token_weight(rule["body"])
    print(f"{rule['name']:13} {rule['glob']:26} {rule['tokens']:>4} tokens")

CHANGED = ["cases/UK-2291/translations/birth-certificate.md"]
picked = activate(CHANGED, loaded)
print(f"\nchanged paths : {CHANGED}")
print(f"activated     : {[r['name'] for r in picked]}")
print(f"always on     : {sum(r['tokens'] for r in loaded)} rule tokens")
print(f"activated     : {sum(r['tokens'] for r in picked)} rule tokens")

appeals       cases/*/appeals/**           28 tokens
biometrics    cases/*/biometrics/**        27 tokens
fees          cases/*/fees/**              23 tokens
medical       cases/*/medical/**           28 tokens
translations  cases/*/translations/**      30 tokens

changed paths : ['cases/UK-2291/translations/birth-certificate.md']
activated     : ['translations']
always on     : 136 rule tokens
activated     : 30 rule tokens


Now run the same six attempts against the activated set instead of the folder.

In [9]:
scoped_text = "\n\n".join(f"## {r['name']}\n{r['body']}" for r in picked)
after = [ask(scoped_text) for _ in range(6)]
after_wrong = [a for a in after if any(w in a[0].lower() for w in OFF_TOPIC)]

print(f"before: {answers[0][1]} prompt tokens, "
      f"{len(wrong)} of {len(answers)} answers followed an irrelevant rule")
print(f"after : {after[0][1]} prompt tokens, "
      f"{len(after_wrong)} of {len(after)} answers followed an irrelevant rule")
print(f"saved : {answers[0][1] - after[0][1]} prompt tokens on every task")

before: 210 prompt tokens, 1 of 6 answers followed an irrelevant rule
after : 103 prompt tokens, 0 of 6 answers followed an irrelevant rule
saved : 107 prompt tokens on every task


## The gate

The regression to stop is somebody adding a rule file and reaching for the folder again. No model, so
it runs on every commit.

In [10]:
import shutil


def test_a_rule_only_loads_for_a_path_it_matches():
    hits = activate(["cases/UK-2291/translations/birth-certificate.md"], loaded)
    names = {r["name"] for r in hits}
    assert names == {"translations"}, f"activated {names}, expected only translations"
    budget = sum(r["tokens"] for r in hits)
    assert budget < sum(r["tokens"] for r in loaded), "scoping the folder saved nothing"


test_a_rule_only_loads_for_a_path_it_matches()
shutil.rmtree(WORKSPACE)
print("gate holds: one changed path activates one rule, and the budget shrinks")

gate holds: one changed path activates one rule, and the budget shrinks


Point `activate` at the whole list instead of the matches and this test fails on the first line. The
temporary rule folder is removed on the way out, so the lesson leaves nothing behind.

### Enterprise exploration

- A case touches four folders at once. Four rules activate and two disagree. Who sets the order, and
  where is that written down?
- Rules are authored by policy staff. What review gate stops a glob that matches everything, and what
  does finding out in production cost?
- The activated set changes per task, so the front of the request changes too. What does that do to
  your cache hit rate, and how would you measure it?
- An auditor asks which rules were in force for case UK-2291 last March. Nothing records that. What
  is the compliance exposure?

### Key takeaways

- A rule carries its own condition. Code reads that condition, not the model.
- Joining every rule into one prompt deletes the scoping and taxes every task.
- Weigh each rule with numbers the provider reports, not with a guess.
- A rule with no scope is always on. Make that an error, not a default.